# Python Environment Setup for EC2

This notebook explains how to:
1. Understand why we install Python 3.11 side-by-side
2. Verify Python versions
3. Create and use virtual environments
4. Install packages correctly

**Important:** On EC2, run terminal commands without the `!` prefix.

## Why Side-by-Side Python Installation?

Corporate Linux AMIs (golden images) often include Python 3.7 as the **system Python**.
This system Python is used by:
- `yum` / `dnf` package managers
- AWS agents (CloudWatch, SSM)
- OS-level scripts

**If you overwrite system Python, these tools break.**

The safe approach:
- Install Python 3.11 to `/usr/local/bin/python3.11` (a separate location)
- Create virtual environments using `python3.11`
- Never redirect `python3` or `python` to point to 3.11 system-wide

In [ ]:
import sys
import platform

print("=== Current Python Info ===")
print(f"Python version:    {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Platform:          {platform.system()} {platform.release()}")
print(f"Architecture:      {platform.machine()}")

## Checking Python Versions on EC2

After installing Python 3.11, verify both versions are available:

In [ ]:
# Run in EC2 terminal:
check_commands = [
    "python3 --version",               # System Python (should be 3.7)
    "which python3",                   # Path of system Python
    "/usr/local/bin/python3.11 --version",  # Our Python 3.11
    "/usr/local/bin/python3.11 -m pip --version",  # pip for Python 3.11
]

for cmd in check_commands:
    print(f"$ {cmd}")

In [ ]:
# Verify current Python (in this notebook environment)
!python3 --version
!which python3

## Installing Python 3.11 on Amazon Linux 2

These commands build Python 3.11 from source on Amazon Linux 2 / RHEL-style systems.
Run them in your EC2 terminal as root.

In [ ]:
# EC2 TERMINAL COMMANDS (do not run in notebook):
install_steps = """
# Step 1: Install build dependencies
yum groupinstall -y "Development Tools"
yum install -y gcc openssl-devel bzip2-devel libffi-devel zlib-devel xz-devel \
    wget make sqlite-devel readline-devel

# Step 2: Download Python 3.11.9 source
cd /usr/src
wget https://www.python.org/ftp/python/3.11.9/Python-3.11.9.tgz
tar xzf Python-3.11.9.tgz
cd Python-3.11.9

# Step 3: Configure and build
./configure --enable-optimizations --with-ensurepip=install
make -j$(nproc)

# Step 4: Install WITHOUT overwriting system Python
make altinstall    # <-- CRITICAL: altinstall, NOT install

# Step 5: Verify
/usr/local/bin/python3.11 --version
python3 --version   # Should still show 3.7.x
"""

print(install_steps)

## Understanding Virtual Environments

A virtual environment (venv) is an **isolated Python installation** for one project.

**Key concept:** `venv` does NOT install Python. It **uses** the Python you point it to.

In [ ]:
# Visual: what's inside a venv folder
venv_structure = """
/opt/apps/YOUR_REPO_NAME/
├── venv/
│   ├── bin/
│   │   ├── python          ← symlink to Python 3.11
│   │   ├── pip             ← pip for Python 3.11
│   │   ├── streamlit       ← streamlit binary (after install)
│   │   └── activate        ← the activation script
│   ├── lib/
│   │   └── python3.11/
│   │       └── site-packages/   ← installed packages go here
│   └── pyvenv.cfg          ← records which Python was used
├── app.py
├── requirements.txt
└── .env
"""
print(venv_structure)

## Creating the Virtual Environment

In [ ]:
# WRONG — uses system Python 3.7
wrong_command = "python3 -m venv venv"

# CORRECT — explicitly uses Python 3.11
correct_command = "/usr/local/bin/python3.11 -m venv venv"

print(f"❌ Wrong:   {wrong_command}")
print(f"✅ Correct: {correct_command}")

print()
print("Full setup flow (EC2 terminal commands):")
print("""
cd /opt/apps/YOUR_REPO_NAME

# Create venv with Python 3.11
/usr/local/bin/python3.11 -m venv venv

# Activate
source venv/bin/activate

# Verify Python version
python --version    # Must show Python 3.11.9
which python        # Must show .../venv/bin/python

# Upgrade pip
pip install --upgrade pip==24.0

# Install packages
pip install -r requirements.txt
pip install streamlit

# Deactivate when done
deactivate
""")

## Verifying the venv Uses Python 3.11

In [ ]:
# After activating venv, run these to verify:
verification = {
    "which python": "Must show /opt/apps/YOUR_REPO_NAME/venv/bin/python",
    "python --version": "Must show Python 3.11.9",
    "which pip": "Must show /opt/apps/YOUR_REPO_NAME/venv/bin/pip",
    "pip --version": "Must reference python 3.11",
}

print("After 'source venv/bin/activate', verify:")
print()
for cmd, expected in verification.items():
    print(f"$ {cmd}")
    print(f"  → {expected}")
    print()

## Useful pip Commands

In [ ]:
pip_commands = [
    ("pip install package",                    "Install a package"),
    ("pip install package==1.2.3",             "Install specific version"),
    ("pip install -r requirements.txt",        "Install from requirements file"),
    ("pip install --upgrade pip==24.0",        "Upgrade pip to specific version"),
    ("pip list",                               "List installed packages"),
    ("pip show streamlit",                     "Show package details"),
    ("pip freeze > requirements.txt",          "Save installed packages to file"),
    ("pip uninstall package",                  "Remove a package"),
    ("pip list --outdated",                    "Show packages with updates available"),
    ("pip install --upgrade setuptools wheel", "Fix common install failures"),
]

print(f"{'Command':<45} {'Purpose'}")
print("-" * 80)
for cmd, purpose in pip_commands:
    print(f"{cmd:<45} {purpose}")

## Checking What's Installed in the Current Environment

In [ ]:
import pkg_resources

installed = sorted([(d.project_name, d.version) for d in pkg_resources.working_set])

print(f"{'Package':<30} {'Version'}")
print("-" * 45)
for name, version in installed[:20]:  # Show first 20
    print(f"{name:<30} {version}")
print(f"... and {len(installed) - 20} more packages" if len(installed) > 20 else "")

## Common Mistakes and Fixes

In [ ]:
mistakes = [
    {
        "mistake": "Created venv with system Python 3.7",
        "symptom": "python --version shows 3.7 inside venv",
        "fix": "deactivate && rm -rf venv && /usr/local/bin/python3.11 -m venv venv"
    },
    {
        "mistake": "Forgot to activate venv before pip install",
        "symptom": "Packages install to system Python, not venv",
        "fix": "source venv/bin/activate  (then verify: which pip)"
    },
    {
        "mistake": "Used make install instead of make altinstall",
        "symptom": "System Python symlink overwritten",
        "fix": "Restore symlink: ln -sf /usr/bin/python3.7 /usr/bin/python3"
    },
    {
        "mistake": "Committed venv/ to GitHub",
        "symptom": "Huge repository, venv appears in git status",
        "fix": "echo 'venv/' >> .gitignore && git rm -r --cached venv/"
    },
]

for i, m in enumerate(mistakes, 1):
    print(f"Mistake {i}: {m['mistake']}")
    print(f"  Symptom: {m['symptom']}")
    print(f"  Fix:     {m['fix']}")
    print()